# The Evolution of Efficient Attention

> GPT-2 used a 1K context; modern long-context models may advertise 1M. As context grows, Attention processes more token pairs, and this cost grows quadratically.
>
> Engineers have reduced the bill along three paths: calculate fewer pairs with Sparse Attention, change the order of computation with Linear Attention, or replace the memory mechanism with SSMs and Mamba. Since 2024, these apparently separate paths have begun to converge.
>
> We follow the historical sequence. Each method ends with a limitation that motivates the next. The goal is to place Mamba, Lightning Attention, NSA/DSA, and hybrid architectures on the correct path and understand the problem each solves.


## 1. Longer Context Makes Attention More Expensive

Attention lets every token inspect every historical token, producing $N \times N$ pairs. We first calculate concrete matrix sizes from GPT-2-era contexts through modern long-context settings.


In [ ]:
# The O(N²) bottleneck of softmax attention: hand-compute a few concrete numbers

d = 128  # head_dim

print(f"head_dim = {d}")
print(f"{'seq len N':<15} {'QK^T elements':<20} {'FP16 memory':<15} {'vs 4K'}")
print("-" * 60)
base_elems = None
for N in [4096, 8192, 32768, 131072, 524288]:
    elems = N * N
    bytes_ = elems * 2  # FP16
    gb = bytes_ / (1024**3)
    if base_elems is None:
        base_elems = elems
    label = f"{N//1024}K" if N >= 1024 else str(N)
    print(f"{label:<15} {elems:<20,} {gb:<15.2f} {elems/base_elems:.0f}x")

print()
print("Key observation: from 4K to 512K, the sequence length grows 128x, but the attention intermediate grows 16384x")
print("This is why long context cannot use pure softmax attention — a single layer's intermediate result is hundreds of GB")


Increasing context from 1K to 128K multiplies length by 128 but the Attention matrix by about 16,000. At 1M, it is one million times the 1K matrix. $O(N^2)$ is the shape of a real bill.

KV Cache grows linearly with sequence length, so long context suffers quadratic Attention compute and linear cache memory. This chapter focuses on the first cost, beginning with the simplest response: compute fewer pairs.


## 2. Path One: Sparse Attention

Methods such as Longformer and BigBird appeared around 2020 with a direct idea: if all token pairs are too expensive, skip some. The central question is **which pairs to skip**. We begin by measuring how sparse softmax Attention weights already are.


### 2.1 Sparsity of Attention Distributions

First, quantify the observation that "most attention weights are near zero". Below, a random attention matrix is generated and we look at the softmax distribution.


In [ ]:
# Measure: how sparse is the softmax attention weight distribution in practice

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
N, d = 128, 64

Q = torch.randn(N, d)
K = torch.randn(N, d)

# Standard softmax attention (with causal mask)
scores = Q @ K.T / (d ** 0.5)
# causal mask
mask = torch.triu(torch.ones(N, N), diagonal=1).bool()
scores = scores.masked_fill(mask, float('-inf'))
weights = F.softmax(scores, dim=-1)

# Statistics on each query's weight distribution
print(f"Attention matrix shape: {weights.shape}")
print(f"Statistics per row (each query's weight distribution):")
print()

# Take query 100 (sees 100 history tokens)
q_idx = 100
row = weights[q_idx, :q_idx+1]  # only look within the causal range

sorted_row, _ = row.sort(descending=True)
top_5_sum = sorted_row[:5].sum().item()
top_10_sum = sorted_row[:10].sum().item()
top_20_sum = sorted_row[:20].sum().item()

print(f"Query {q_idx} sees {q_idx+1} keys (causal)")
print(f"  Top-5  keys hold: {top_5_sum*100:.1f}% of total weight")
print(f"  Top-10 keys hold: {top_10_sum*100:.1f}% of total weight")
print(f"  Top-20 keys hold: {top_20_sum*100:.1f}% of total weight")
print(f"  The remaining {q_idx+1-20} keys together: {(1-top_20_sum)*100:.1f}%")
print()
print(f"Key observation: the top-20 keys take {top_20_sum*100:.0f}% of the weight")
print(f"Computing the remaining {q_idx+1-20} keys barely changes the output — this is the opening for sparse attention")

# Visualization: one query's weight distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(q_idx+1), row.numpy(), color='#3498db')
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Attention weight')
axes[0].set_title(f'Query {q_idx} attention weights\nmost near 0, a few keys dominate')
axes[0].grid(True, alpha=0.3)

# Cumulative weight
cumulative = row.sort(descending=True)[0].cumsum(0).numpy()
axes[1].plot(range(1, q_idx+2), cumulative, '-o', markersize=3, color='#e74c3c')
axes[1].axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='90% weight')
axes[1].axhline(y=0.95, color='gray', linestyle=':', alpha=0.5, label='95% weight')
axes[1].set_xlabel('Top-K keys')
axes[1].set_ylabel('Cumulative weight')
axes[1].set_title('Cumulative weight: a few keys cover 90%+ of the total')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 2.2 Fixed Patterns

The simplest sparse attention uses a **fixed pattern** to decide which keys to see. Common choices:

| Pattern | How keys are chosen | Representative |
|:---|:---|:---|
| **Sliding Window** | each query only sees the W nearest keys | Longformer, Mistral |
| **Local + Global** | sliding window + a few global tokens (always seen) | Longformer, Big Bird |
| **Dilated Sliding** | strided, to widen the receptive field | Longformer |
| **Block-sparse** | split the sequence into blocks, compute attention only between some blocks | Sparse Transformer |

Fixed patterns are simple, interpretable, and easy to implement. The long-context section in Part 4 already covered the specifics of Sliding Window.

The problem is inflexibility — truly important keys may be far from the current query (for retrieval, cross-segment coreference), and a fixed pattern will miss them. Next we see how learnable sparse attention fixes this.


### 2.3 Learnable Sparse Attention

The core idea of learnable sparse attention: add a lightweight "router" that scores all keys for each query, picks the top-N important ones, and only runs full attention on those N.

The implementation usually looks like:

```
1. Use a lightweight MLP to compute an importance score for each key
   score[i] = MLP(query, key_i)  → scalar

2. Pick the top-N important keys
   selected_keys = topk(scores, N)

3. Full softmax attention only on the selected keys
   output = softmax(Q @ selected_K.T) @ selected_V
```

The first step uses an MLP rather than full attention — the MLP's complexity is much lower than attention (a few GFLOPs per forward). After filtering out most keys, the full attention becomes worthwhile.

DeepSeek's DSA (DeepSeek Sparse Attention, introduced in V3.2) follows exactly this idea: an ultra-light indexer (MQA-style mini attention + ReLU activation) scores each key, picks top-N, then runs full attention. NSA goes further; the next section expands on it.


Fixed sliding-window and block patterns are simple and predictable, but people decide which pairs matter. Long-range references are precisely what such rules may miss. Learned sparsity delegates selection to the model, but irregular patterns can be difficult to execute efficiently on GPUs.

This motivates a more radical path: skip no pairs conceptually, but reorder the algebra so the $N \times N$ matrix never materializes.


## 3. Path Two: Linear Attention

“Transformers are RNNs” (2020) observed that standard Attention can be written as $(\phi(Q)\phi(K)^T)V$. If softmax is replaced by a separable feature function, associativity lets us compute $\phi(Q)(\phi(K)^T V)$. The intermediate matrix changes from $N \times N$ to $d \times d$. With sequence length in the hundreds of thousands and head dimension near 64 or 128, complexity becomes linear in length.


### 3.1 Linear Attention Computation Order

The key observation of linear attention: standard attention, written as $\text{softmax}(Q K^T) V$, computes $Q K^T$ first (giving the $N \times N$ matrix), then multiplies by $V$. If we replace softmax with a separable form $\phi(Q) \phi(K)^T$, we can move the parentheses:

$$
\underbrace{(\phi(Q) \phi(K)^T)}_{N \times N} V \quad \Rightarrow \quad \phi(Q) \underbrace{(\phi(K)^T V)}_{d \times d}
$$

New order of computation: first compute $\phi(K)^T V$ (a small $d \times d$ matrix), then left-multiply by $\phi(Q)$.

| Order | Intermediate size | Total FLOPs |
|:---|:---|:---|
| Standard $(Q K^T) V$ | $N \times N$ | $O(N^2 d)$ |
| Linear $Q (K^T V)$ | $d \times d$ | $O(N d^2)$ |

When $d \ll N$ (say $d=128$, $N=32K$), going from $O(N^2 d)$ to $O(N d^2)$ is a huge saving.

But softmax cannot be written in the separable form $\phi(Q) \phi(K)^T$ — it normalizes each row across all keys. Linear attention instead replaces softmax with other kernel functions; common choices include:
- $\phi(x) = \text{elu}(x) + 1$ (first proposed by Katharopoulos et al. 2020)
- $\phi(x) = \text{ReLU}(x)$ (simpler but slightly worse)
- Approximations based on random features

Below, a 4-token example hand-computes both softmax attention and linear attention side by side.


In [ ]:
# Hand comparison: softmax attention vs linear attention (4-token example)

import torch
import torch.nn.functional as F

torch.manual_seed(42)

N = 4   # 4 tokens
d = 3   # head_dim = 3

Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

print("Q =\n", Q)
print("\nK =\n", K)
print("\nV =\n", V)

# === standard softmax attention ===
scores = Q @ K.T / (d ** 0.5)
attn_weights = F.softmax(scores, dim=-1)
out_softmax = attn_weights @ V
print("\n=== Softmax Attention ===")
print(f"QK^T / sqrt(d) =\n{scores}")
print(f"\nsoftmax(QK^T) =\n{attn_weights}")
print(f"\noutput =\n{out_softmax}")

# === Linear attention ===
# φ(x) = elu(x) + 1
phi_Q = F.elu(Q) + 1
phi_K = F.elu(K) + 1

# Standard form (O(N²d)): first φ(Q)φ(K)^T, then multiply by V
linear_scores_naive = phi_Q @ phi_K.T
# Normalize: each row divided by the sum of its denominators
normalizer = linear_scores_naive.sum(dim=-1, keepdim=True)
out_linear_naive = (linear_scores_naive @ V) / normalizer

print("\n=== Linear Attention ===")
print(f"φ(Q) = elu(Q)+1 =\n{phi_Q}")
print(f"\nφ(K) = elu(K)+1 =\n{phi_K}")
print(f"\nφ(Q)φ(K)^T =\n{linear_scores_naive}")
print(f"\noutput =\n{out_linear_naive}")

# === Linear attention, O(Nd²) form ===
# Use associativity: φ(Q) @ (φ(K)^T @ V)
KV = phi_K.T @ V  # [d, d] small matrix
out_linear_fast = phi_Q @ KV / normalizer
print(f"\n=== Linear Attention (associativity, regrouped) ===")
print(f"φ(K)^T @ V =\n{KV}")
print(f"\noutput (should match the above) =\n{out_linear_fast}")
print(f"\nDo both results match: {torch.allclose(out_linear_naive, out_linear_fast, atol=1e-6)}")


### 3.2 Limitations of Linear Attention

Linear attention is mathematically an approximation of softmax attention. **Their outputs are not exactly the same** — here is why.

A key property of softmax is **sharpness**: when one query-key dot product is much larger than the others, softmax puts almost all the weight on that key. For example:

```
softmax([10, 1, 1, 1]) ≈ [0.999, 0.0003, 0.0003, 0.0003]  ← extremely sharp
```

This "almost one-hot" weighting lets attention precisely aggregate a specific token's information onto the query — essential for tasks like retrieval and copy.

Linear attention has no such sharpness. $\phi(Q) \phi(K)^T$ is an element-wise non-linear product, and the gap between the largest and smallest values is much smaller than with softmax. As a result, the weight distribution is relatively flat and cannot give a very high weight to any specific token. The code below quantifies the comparison.


In [ ]:
# Sharpness comparison: softmax vs linear attention

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Build a scene: query is very similar to key[0], and dissimilar to other keys
torch.manual_seed(0)
N, d = 8, 16

# query ≈ K[0], orthogonal to other K
K = torch.randn(N, d)
Q = K[0:1].clone() + 0.01 * torch.randn(1, d)  # Q ≈ K[0]
V = torch.eye(N)  # identity matrix, so the output is the attention weights themselves

# Softmax attention
scores_softmax = Q @ K.T / (d ** 0.5)
weights_softmax = F.softmax(scores_softmax, dim=-1)

# Linear attention
phi_Q = F.elu(Q) + 1
phi_K = F.elu(K) + 1
linear_scores = phi_Q @ phi_K.T
weights_linear = linear_scores / linear_scores.sum(dim=-1, keepdim=True)

print(f"Query ≈ Key[0], ideal weight should go almost entirely to position 0")
print()
print(f"{'Position':<10} {'Softmax weight':<20} {'Linear weight':<20}")
print("-" * 50)
for i in range(N):
    print(f"{i:<10} {weights_softmax[0, i].item():<20.4f} {weights_linear[0, i].item():<20.4f}")

print()
print(f"Key observations:")
print(f"  Softmax puts {weights_softmax[0, 0].item()*100:.1f}% of the weight on position 0 → almost one-hot")
print(f"  Linear only puts {weights_linear[0, 0].item()*100:.1f}% of the weight on position 0 → flat distribution")
print(f"  → Linear attention is inherently weak at precise retrieval")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(N), weights_softmax[0].numpy(), color='#e74c3c')
axes[0].set_title('Softmax attention weights\n(sharp, retrieval-friendly)')
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Weight')
axes[0].set_ylim(0, 1)

axes[1].bar(range(N), weights_linear[0].numpy(), color='#3498db')
axes[1].set_title('Linear attention weights\n(flat, retrieval-weak)')
axes[1].set_xlabel('Key position')
axes[1].set_ylabel('Weight')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()


The reordered computation loses softmax's sharp winner-take-most behavior, so Linear Attention performs worse when it must retrieve one exact token. Precise retrieval is one of the central requirements of long context.

Another community reached a similar limitation from the older RNN idea of constant-size recurrent memory.


## 4. Path Three: SSMs and Mamba

An RNN updates a fixed-size hidden state for each token, so memory is constant regardless of sequence length. Classic RNNs struggle with long-range dependencies and are slow to train because recurrence is sequential.

SSMs (State Space Models), including S4 (2021), use structured state equations that permit parallel training, addressing the second problem. The first remains: every input follows the same transition rule, so the model cannot choose what to remember from content. Mamba (2023) makes the parameters input-dependent, adding **selectivity**.


### 4.1 The Recurrent Form of Linear Attention

Linear Attention has a vectorized form and an associatively reordered form. A third equivalent form updates a hidden state one token at a time, revealing its connection to recurrent sequence models.


In [ ]:
# The recurrent form of Linear Attention (causal version)

import torch
import torch.nn.functional as F

torch.manual_seed(42)

N, d = 5, 3
Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

phi_Q = F.elu(Q) + 1
phi_K = F.elu(K) + 1

# Recurrent form (causal): S_t = S_{t-1} + φ(k_t) v_t^T, o_t = φ(q_t) S_t
# each token only sees tokens up to itself
S = torch.zeros(d, d)
out_rec = []
print("=== Linear Attention unrolled recurrence (causal) ===")
for t in range(N):
    S = S + phi_K[t].unsqueeze(1) @ V[t].unsqueeze(0)  # accumulate up to t
    o_t = phi_Q[t] @ S
    out_rec.append(o_t)
    print(f"t={t}: after accumulation S norm = {S.norm().item():.3f}, o_t[0] = {o_t[0].item():.3f}")
out_rec = torch.stack(out_rec)

# Compare: vectorized form with a causal mask (each query only sees previous keys)
scores = phi_Q @ phi_K.T  # [N, N]
# causal mask: upper triangle (j > i) set to 0
causal_mask = torch.tril(torch.ones(N, N))
scores_causal = scores * causal_mask
normalizer = scores_causal.sum(dim=-1, keepdim=True).clamp(min=1e-6)
out_vec_causal = (scores_causal @ V) / normalizer

# Unnormalized recurrence vs unnormalized vectorized (compare φ(q)S form directly)
out_vec_unnorm = (scores_causal @ V)  # [N, d]

print()
print(f"Do recurrence and vectorized (causal) match: "
      f"{torch.allclose(out_rec, out_vec_unnorm, atol=1e-6)}")
print()
print("Key observation: the recurrent form is naturally causal (each token only sees history)")
print("Unrolling the recurrence gives causal Linear Attention; per step O(d²), total O(Nd²)")


### 4.2 SSM State Equations

The State Space Model (SSM) grew out of control theory and writes sequence modeling as "state + input + output". The discrete-time equations are:

$$
x_t = A x_{t-1} + B u_t, \quad y_t = C x_t
$$

where $u_t \in \mathbb{R}^1$ is the current input signal (a scalar, like one feature channel of a token), $x_t \in \mathbb{R}^N$ is the N-dimensional hidden state (analogous to the LSTM cell state), and $y_t$ is the output. $A \in \mathbb{R}^{N \times N}$ is the state transition matrix, $B \in \mathbb{R}^{N \times 1}$ projects the input into the state, and $C \in \mathbb{R}^{1 \times N}$ projects the state back to the output.

Compared with Linear Attention: the recurrence of Linear Attention is $S_t = S_{t-1} + \phi(k_t) v_t^T$, equivalent to $A = I$ (identity accumulation), $B = \phi(k_t)$, $C = \phi(q_t)$, with $B, C$ varying over time. The key difference in an SSM is a **learned, fixed** matrix $A$ that makes the state evolve according to some dynamics, rather than simply accumulating.

The role of $A$ becomes clear from two extremes:

- $A = I$: the state only grows, never decays — long memory but tends to explode
- $A = 0$: the state resets every step — only sees the current input
- When the eigenvalues of $A$ are less than 1, the state decays exponentially — only recent information survives, like a "memory half-life"

In practice, SSM training initializes $A$ with a special structure (e.g. the HiPPO matrix) so that memory stays stable over long distances. Below, a 4-step hand computation makes the recurrence clear.


In [ ]:
# Hand-compute an SSM: 4-step recurrence, observe the effect of A on memory

import torch

# Tiny config: 2-dim state, scalar input
A = torch.tensor([[0.5, 0.0],
                  [0.0, 0.9]])  # diagonal A, two channels with different decay rates
B = torch.tensor([[1.0], [1.0]])
C = torch.tensor([[1.0, 1.0]])

# Input sequence: u = [3, 0, 0, 0], observe how the 3 written at t=0 decays
u_seq = [3.0, 0.0, 0.0, 0.0]

x = torch.zeros(2, 1)  # initial hidden state
print("=== 4-step SSM by hand ===")
print(f"A = diag(0.5, 0.9), B = [1, 1]^T, C = [1, 1]")
print(f"input u = {u_seq}")
print()
for t, u in enumerate(u_seq):
    x = A @ x + B * u           # state update
    y = (C @ x).item()          # output
    print(f"t={t}: u={u}")
    print(f"      x = [{x[0,0].item():.4f}, {x[1,0].item():.4f}]")
    print(f"      y = {y:.4f}")
    print()

print("Key observations:")
print("  - Channel 0 (decay 0.5): wrote 3 at t=0, after 3 decays only 0.375 left → short-term memory")
print("  - Channel 1 (decay 0.9): wrote 3 at t=0, after 3 decays 2.187 left → long-term memory")
print("  - Different eigenvalues of A let the SSM maintain multiple time scales at once")


### 4.3 Selective SSM

Vanilla SSMs have a structural weakness: $A, B, C$ are fixed parameters, applying the same transition rule to all time steps and all inputs. As a result, the model cannot **selectively remember or forget based on input content** — it struggles with tasks that require "remember this specific token".

Mamba (2023) makes $B, C$ and the discretization step $\Delta$ all functions of the input:

$$
B_k = f_B(u_k), \quad C_k = f_C(u_k), \quad \Delta_k = f_\Delta(u_k)
$$

The discretized state update becomes $x_k = \bar{A}_k x_{k-1} + \bar{B}_k u_k$, where $\bar{A}_k, \bar{B}_k$ are determined by $\Delta_k$ (via zero-order-hold discretization $\bar{A} = \exp(\Delta A)$, $\bar{B} = (\Delta A)^{-1}(\exp(\Delta A) - I) \Delta B$). The three input-dependent quantities carry different semantics:

- **$\Delta_k$ controls the time scale**: it decides "how strong this step's state update is". A large $\Delta_k$ lets new input dominate (overwrite old memory); a small one preserves history. Equivalent to the LSTM forget gate.
- **$B_k$ controls what to write**: how the current input is projected into the state space. Equivalent to the input gate.
- **$C_k$ controls what to read**: how to read out from the state space. Equivalent to the output gate.

$A$ is still a learned fixed matrix, but through the modulation of $\Delta_k$, the effective transition $\bar{A}_k$ also varies with the input. This input-dependent mechanism gives the SSM, for the first time, a "content-based routing" ability close to attention. Below, a mini PyTorch demo shows how $B_k, C_k, \Delta_k$ vary with the input.


In [ ]:
# Mamba selective SSM: B/C/Δ are functions of the input

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

d_model = 8       # input dimension
d_state = 4       # hidden state dimension N

# Projection functions: map the input u_k to B_k, C_k, Δ_k
proj_B = nn.Linear(d_model, d_state)
proj_C = nn.Linear(d_model, d_state)
proj_Delta = nn.Linear(d_model, 1)  # one scalar Δ per channel

# Fixed A matrix (in practice initialized as a diagonal of learnable parameters)
A_log = torch.randn(d_state) * 0.5  # log-parameterization to keep A negative

def selective_ssm_forward(u_seq):
    """Simplified selective SSM forward, returns output at each time step"""
    N = u_seq.shape[0]
    # Input-dependent B, C, Δ
    B_seq = proj_B(u_seq)        # [N, d_state]
    C_seq = proj_C(u_seq)        # [N, d_state]
    Delta_seq = F.softplus(proj_Delta(u_seq)).squeeze(-1)  # [N], guaranteed > 0

    A = -torch.exp(A_log)        # A < 0, ensures stability
    x = torch.zeros(d_state)
    outputs = []

    for t in range(N):
        # Discretization (zero-order hold, scalarized Δ)
        d_t = Delta_seq[t]
        A_bar = torch.exp(A * d_t)             # for diagonal A, exp(ΔA) is element-wise
        B_bar = (d_t) * B_seq[t]               # simplified: ignore the exact (ΔA)^{-1}(exp(ΔA)-1) term
        # State update
        x = A_bar * x + B_bar * u_seq[t, 0]    # use the first channel of u as the scalar input
        # Output
        y = C_seq[t] @ x
        outputs.append(y.item())

    return outputs, Delta_seq, B_seq, C_seq

# Feed a sequence and watch how Δ_k changes across tokens
u_seq = torch.randn(6, d_model)
outputs, Delta, B, C = selective_ssm_forward(u_seq)

print("=== Selective SSM: B/C/Δ vary with the input ===")
print(f"input sequence shape: {u_seq.shape}")
print()
print(f"Δ_k (time scale) per token:")
for t in range(u_seq.shape[0]):
    print(f"  t={t}: Δ={Delta[t].item():.4f}  "
          f"B={B[t].tolist()[:3]}...  C={C[t].tolist()[:3]}...")

print()
print("Key observations:")
print("  - Δ_k differs per step → the model can decide when to overwrite the state based on content")
print("  - B_k controls the write direction → how the current token is encoded into the state")
print("  - C_k controls the read direction → what the current token reads from the state")
print("  - This input-dependent behavior is the fundamental reason Mamba beats vanilla SSMs")


### 4.4 What Selectivity Changes

What essential improvement does the selective mechanism bring? A classic benchmark in the Mamba paper is the **selective copy task**. Given the input sequence `[3, 1, 4, 1, 5]`, the model must, during the output phase, reproduce all the `1`s in order (or some other rule-selected subset). The key is that the model must **distinguish by content** which tokens to remember and which to ignore — not by fixed position (first 3, last 5).

A vanilla SSM (fixed $A/B/C$) cannot do this task: all positions go through the same state update, so the model cannot tell "the 1 at position 1" apart from "the 1 at position 3" — they are processed by the same dynamics in state space. Linear Attention has the same limitation: the accumulation of $\phi(k_t) v_t^T$ uses the same $\phi$ for every token.

Selective SSMs can do this task because $B_k, C_k, \Delta_k$ vary with the input. The model can learn to "give a large $\Delta$ when it sees a 1" (strong write) and "a small $\Delta$ for other tokens" (preserve memory). This is equivalent to learning a "when to pay attention" policy. Below, a minimal contrast experiment shows the difference: a fixed-parameter SSM treats every token equally, while the selective version can modulate write strength based on content.


In [ ]:
# Compare fixed SSM vs selective SSM: which one can "selectively remember"

import torch

torch.manual_seed(42)

# Input sequence: [3, 1, 4, 1, 5]
# Task goal: only remember the cumulative value of all 1s; other tokens should be filtered out
tokens = [3.0, 1.0, 4.0, 1.0, 5.0]
target_sum = sum(t for t in tokens if t == 1.0)  # should be 2.0
d_state = 4

# === Plan A: fixed-parameter SSM ===
A_fixed = torch.tensor([[0.9, 0.0, 0.0, 0.0],
                        [0.0, 0.8, 0.0, 0.0],
                        [0.0, 0.0, 0.7, 0.0],
                        [0.0, 0.0, 0.0, 0.6]])
B_fixed = torch.tensor([1.0, 1.0, 1.0, 1.0])  # all tokens written with the same B

x = torch.zeros(d_state)
fixed_final = None
for u in tokens:
    x = A_fixed @ x + B_fixed * u  # regardless of what u is, B stays the same
    fixed_final = x.tolist()

print("=== Fixed-parameter SSM ===")
print(f"Task: only accumulate token=1 values, expect the final state to reflect sum=2.0")
print(f"Actual final state: {[f'{v:.3f}' for v in fixed_final]}")
print(f"The state mixed in the cumulative value of 3 + 4 + 5 = 12 → cannot filter")
print("→ A fixed B means all tokens are written with the same rule; the model cannot say 'remember only 1'")
print()

# === Plan B: selective SSM ===
# Suppose the model learns Δ_k = f(u_k): large Δ for u=1 (strong write), small Δ otherwise
def get_delta(token):
    if token == 1.0:
        return 2.0  # large Δ: strong write
    return 0.05     # small Δ: weak write, state barely updates

A_diag = torch.tensor([0.9, 0.8, 0.7, 0.6])
B = torch.tensor([1.0, 1.0, 1.0, 1.0])

x = torch.zeros(d_state)
sel_final = None
for u in tokens:
    d = get_delta(u)
    A_bar = torch.exp(torch.log(A_diag) * d)  # effective A
    x = A_bar * x + (d * B) * u                # effective B modulated by Δ
    sel_final = x.tolist()

print("=== Selective SSM ===")
print(f"Δ strategy: u=1 → Δ=2.0 (strong write), other → Δ=0.05 (weak write)")
print(f"Actual final state: {[f'{v:.3f}' for v in sel_final]}")
print(f"Target cumulative = {target_sum}")
print("→ By modulating write strength through Δ, the selective SSM can remember selectively based on content")
print("→ This is the fundamental mechanism that makes Mamba stronger than vanilla SSMs")


Selectivity lets an SSM choose what to remember, but a fixed-size state still cannot preserve unlimited token-level detail. Exact retrieval encounters the same ceiling. The three paths have reached related limits and now begin to merge.


## 5. Convergence of the Three Paths

Two developments since 2024 joined these threads. Mathematically, Mamba-2 showed a structural duality between selective SSMs and Linear Attention—two views of a related model. Operationally, production systems increasingly mix softmax Attention with SSM or linear layers, allowing each to handle what it does best.

Sparse Attention also returned in a learned form. DeepSeek's NSA and DSA replace hand-designed masks with learned selection, addressing the weakness of first-generation sparse patterns.


### 5.1 Duality Between SSMs and Linear Attention

Mamba-1 (selective SSM) solved the expressiveness problem but left an engineering pain: the recurrent form cannot be trained efficiently in parallel. Just like Linear Attention, each step's state update depends on the previous step, so backpropagation has to unroll over time, which is slow for long sequences.

Mamba-2 (2024) makes the core contribution of **State Space Duality (SSD)**: it proves mathematically that selective SSMs and Linear Attention are equivalent, and that a selective SSM can be written in matrix form $\text{SSM}(A, B, C) = C \cdot (\text{the lower-triangular-masked cumulative product of } A) \cdot B$. This matrix form lets the SSM use Linear Attention's parallel training algorithm (one big matrix multiplication) for the forward pass, while switching back to the recurrent form at inference to keep O(1) per step.

Concretely, SSD writes the selective SSM as:

$$
Y = M \cdot (V), \quad M_{ij} = C_i^T \left(\prod_{k=j+1}^{i} A_k\right) B_j
$$

where $M$ is an $N \times N$ "SSM matrix" — structurally similar to an attention matrix (lower-triangular / causal) with weights determined by the learned $A_k, B_k, C_k$. This gives Mamba-2 a key engineering advantage: it can reuse Linear Attention's efficient training infrastructure (e.g. chunk-wise computation, FlashAttention-style kernel fusion).

To summarize, SSD matters in two ways: theoretically, it unifies the SSM and Linear Attention research lines; in engineering, it boosts Mamba's training throughput by 2-8x over Mamba-1. This is why essentially all industrial SSM implementations after 2024 are based on the Mamba-2 SSD framework.


### 5.2 NSA and DSA

The core innovation of NSA (Native Sparse Attention, DeepSeek 2025) is to build sparse attention as **three parallel branches with gated fusion**, each branch solving a different problem.

| Branch | What it does | Problem it solves |
|:---|:---|:---|
| **Compress** | compress history tokens into a low-resolution summary, run coarse attention | distant overall context (no need for per-token precision) |
| **Selection** | based on coarse attention scores, pick top-N important tokens and run fine attention | key distant tokens (need precise attention) |
| **Sliding** | keep a sliding window (the most recent W tokens) | neighboring tokens (local dependency) |

The outputs of the three branches are fused by a gate: $\text{out} = g_1 \cdot \text{compress\_out} + g_2 \cdot \text{select\_out} + g_3 \cdot \text{sliding\_out}$. The gate weights $g_1, g_2, g_3$ are learned by the model itself — think of it as "which branch should this layer trust for the current query".

The key engineering contribution of NSA is that the **whole pipeline is end-to-end trainable** — compress, select, sliding, and gating are all differentiable and can be trained directly by backpropagation. This sounds obvious, but many earlier sparse-attention works (using top-k, hard selection) are non-differentiable and require RL or special gradient estimators.

DSA (DeepSeek Sparse Attention, V3.2) is a simplified NSA — it drops the compress branch and uses only an indexer to pick top-N. The structure is simpler and easier to deploy, but slightly less expressive. DeepSeek-V3.2 replaces the original dense attention with DSA, holding performance on long-context benchmarks while cutting compute sharply.

Below is a minimal NSA implementation showing the three-branch fusion mechanism.


In [ ]:
# Minimal NSA: three parallel branches + gated fusion

import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleNSA(nn.Module):
    """Teaching version of NSA, shows the three-branch parallel structure

    Simplifications:
    - Compress branch: simulated with mean pooling, averaging every chunk_size tokens into 1
    - Selection branch: uses the compress branch's attention scores to pick top-N
    - Sliding branch: fixed window W
    - Gate: the query learns 3 weights
    """
    def __init__(self, d_model, chunk_size=4, top_n=8, window_size=8):
        super().__init__()
        self.d_model = d_model
        self.chunk_size = chunk_size
        self.top_n = top_n
        self.window_size = window_size

        # Each branch has its own value projection
        self.W_V_compress = nn.Linear(d_model, d_model, bias=False)
        self.W_V_select = nn.Linear(d_model, d_model, bias=False)
        self.W_V_sliding = nn.Linear(d_model, d_model, bias=False)

        # Shared query projection (simplification)
        self.W_Q = nn.Linear(d_model, d_model, bias=False)

        # Gate: from the query, produce 3 weights
        self.gate = nn.Linear(d_model, 3)

    def forward(self, x):
        """
        x: [batch, seq_len, d_model]
        return: [batch, seq_len, d_model]
        """
        B, S, D = x.shape
        Q = self.W_Q(x)  # [B, S, D]

        # === Compress branch ===
        # Average every chunk_size tokens into 1
        pad = (self.chunk_size - S % self.chunk_size) % self.chunk_size
        x_padded = F.pad(x, (0, 0, 0, pad)) if pad > 0 else x
        x_chunks = x_padded.view(B, -1, self.chunk_size, D).mean(dim=2)  # [B, S/chunk, D]
        # Simplified: compress branch attends directly with K from mean pooling
        compress_V = self.W_V_compress(x_chunks)  # [B, S/chunk, D]
        compress_scores = Q @ x_chunks.transpose(-1, -2) / (D ** 0.5)
        compress_attn = F.softmax(compress_scores, dim=-1)
        compress_out = compress_attn @ compress_V  # [B, S, D]

        # === Selection branch ===
        # Use the compress branch's attention scores to find important chunks, then go back and pick top-N tokens
        select_V = self.W_V_select(x)  # [B, S, D]
        # Simplified: directly use Q·x for top-N
        select_scores = Q @ x.transpose(-1, -2) / (D ** 0.5)  # [B, S, S]
        # Causal mask
        mask = torch.triu(torch.ones(S, S), diagonal=1).bool()
        select_scores_masked = select_scores.masked_fill(mask, float('-inf'))
        top_n = min(self.top_n, S)
        topk_vals, topk_idx = select_scores_masked.topk(top_n, dim=-1)
        # For each query, run attention on the selected keys
        select_out = torch.zeros(B, S, D)
        for b in range(B):
            for i in range(S):
                idx = topk_idx[b, i]
                sel_scores = Q[b, i] @ x[b, idx].T / (D ** 0.5)
                sel_weights = F.softmax(sel_scores, dim=-1)
                select_out[b, i] = sel_weights @ select_V[b, idx]

        # === Sliding branch ===
        sliding_V = self.W_V_sliding(x)
        sliding_out = torch.zeros(B, S, D)
        for i in range(S):
            start = max(0, i - self.window_size + 1)
            window_x = x[:, start:i+1]  # [B, window, D]
            window_V = sliding_V[:, start:i+1]
            q_i = Q[:, i:i+1]  # [B, 1, D], keep the seq dim
            sel_scores = (q_i @ window_x.transpose(-1, -2)).squeeze(1) / (D ** 0.5)
            sel_weights = F.softmax(sel_scores, dim=-1)
            sliding_out[:, i] = (sel_weights.unsqueeze(-1) * window_V).sum(dim=1)

        # === Gated fusion ===
        gate_logits = self.gate(Q)  # [B, S, 3]
        gate_weights = F.softmax(gate_logits, dim=-1)  # [B, S, 3]

        out = (gate_weights[..., 0:1] * compress_out +
               gate_weights[..., 1:2] * select_out +
               gate_weights[..., 2:3] * sliding_out)
        return out

# Test
torch.manual_seed(42)
nsa = SimpleNSA(d_model=16, chunk_size=4, top_n=4, window_size=4)
x = torch.randn(1, 16, 16)
out = nsa(x)
print(f"input shape: {x.shape}")
print(f"output shape: {out.shape}")
print()
print(f"Three-branch config:")
print(f"  Compress: every {nsa.chunk_size} tokens -> 1 → low-res view for long sequences")
print(f"  Select: top-{nsa.top_n} tokens per query → precise but few")
print(f"  Sliding: the most recent {nsa.window_size} tokens → local dependency")
print(f"  Gate: the query decides the mix ratio of the three branches")


#### End-to-End Training of NSA

Why does NSA emphasize "end-to-end trainable"?

In early sparse attention work, top-K selection is **non-differentiable** — `torch.topk` returns discrete indices, and indices are a discrete operation that backprop cannot pass through. As a result, "which keys to pick" cannot be trained by gradients; the options are:
1. Train the selection policy with RL (reinforcement learning)
2. Use gradient estimators like Gumbel-softmax
3. Fix the selection policy with heuristic rules

NSA's key contribution is to make selection **differentiable**. Specifically, the attention scores from the compress branch are used as "soft selection weights" — instead of hard top-N, it weights all tokens by attention scores (but the scores themselves go through a sparsification step). This lets the whole pipeline be trained by standard backprop.

DSA uses a similar strategy — the indexer's output scores are continuous and differentiable, and the top-N selection uses a straight-through estimator (hard top-N in the forward pass, soft gradients in the backward pass).

In engineering terms, this means NSA and DSA can **directly replace dense attention** without changing the rest of the model, and can be plugged in from the pretraining stage. There is no need to train a dense model first and then convert. This is why they made it into production models like DeepSeek-V3.2.


#### DSA

DSA (DeepSeek Sparse Attention, introduced in V3.2-Exp) takes another road. Unlike NSA's "three parallel branches", DSA does one thing — use a lightweight indexer to score every key, pick top-N, then run full attention.

DSA's indexer is extremely lightweight: a small MQA-style attention with ReLU activation. For each query it computes an "importance score" for every key, picks top-N, and discards the rest. It then runs **standard softmax attention** on the selected top-N — exactly like dense attention, just on a smaller key set.

Compared with NSA:

| Dimension | NSA | DSA |
|:---|:---|:---|
| Branches | 3 (compress + select + sliding) | 1 (indexer + top-N) |
| Complexity | Medium (all three branches run) | Low (only indexer + top-N attention) |
| Expressiveness | High (multiple branches complement) | Medium (single branch) |
| Training difficulty | High (three-branch gating needs tuning) | Low (simple structure) |
| Deployment difficulty | High (complex implementation) | Low (easy to integrate) |

DeepSeek-V3.2 chose DSA over NSA mostly for engineering reasons — DSA's simplicity makes it easier to integrate into the existing MLA inference stack. The FlashMLA library provides sparse kernels, reaching 640 TFlops prefill / 410 TFlops decode on H800.

We will not expand on implementation details here. The key takeaway: DSA = lightweight indexer scoring + top-N selection + standard attention. More concise than NSA, more efficient than dense attention.


### 5.3 Hybrid Architectures

Linear Attention and SSMs share a weakness: less precise retrieval. Linear Attention loses sharp softmax selection, while recurrent state compression in Mamba can blur details over long distances. SSM layers, however, are effective for long-context modeling, style continuity, and sequence extrapolation. Hybrid models use occasional softmax layers to preserve retrieval and many linear or SSM layers to control cost.


Linear Attention also faces an implementation problem: pure recurrence cannot parallelize training and may underuse the GPU. MiniMax-01's Lightning Attention uses a blockwise parallel form for training and Prefill, then a recurrent form for Decode.

The code below simulates its hybrid ratio: seven linear layers and one softmax layer in every eight layers.


In [ ]:
# Simulate the MiniMax-01 hybrid architecture: 7 lightning + 1 softmax every 8 layers

import matplotlib.pyplot as plt
import numpy as np

num_layers = 80
layer_types = []
for i in range(num_layers):
    # MiniMax-01 design: the last of every 8 layers is softmax
    if (i + 1) % 8 == 0:
        layer_types.append('softmax')
    else:
        layer_types.append('lightning')

# Visualization
fig, ax = plt.subplots(figsize=(14, 3))

colors = []
for lt in layer_types:
    if lt == 'softmax':
        colors.append('#e74c3c')
    else:
        colors.append('#3498db')

ax.bar(range(num_layers), [1]*num_layers, color=colors, edgecolor='black', linewidth=0.3)
ax.set_xlabel('Layer index')
ax.set_ylabel('Type')
ax.set_title(f'MiniMax-01 style hybrid: {num_layers} layers, red=softmax attention, blue=lightning attention')
ax.set_yticks([])

# Annotate the first 16 layers
for i in range(min(16, num_layers)):
    label = 'S' if layer_types[i] == 'softmax' else 'L'
    ax.text(i, 0.5, label, ha='center', va='center', color='white', fontsize=9, fontweight='bold')

ax.set_xlim(-0.5, num_layers - 0.5)
plt.tight_layout()
plt.show()

# Statistics
n_softmax = sum(1 for lt in layer_types if lt == 'softmax')
n_lightning = num_layers - n_softmax
print(f"Total layers: {num_layers}")
print(f"Softmax layers: {n_softmax} ({n_softmax/num_layers*100:.0f}%)")
print(f"Lightning layers: {n_lightning} ({n_lightning/num_layers*100:.0f}%)")
print()
print("Key observation: 10% of softmax attention layers supply retrieval capability")
print("The other 90% of lightning attention layers handle long sequences efficiently")


## 6. Comparing the Three Paths

The following table places the conclusions we calculated or ran side by side. Exercise 1 implements the vectorized Linear Attention form and checks its complexity.


### 6.1 Complexity Summary

Put the structures from this section together and compare training vs inference complexity. Training looks at total FLOPs (parallelizable); inference looks at the cost of generating one token (cannot avoid the recurrent nature of KV cache or state update).

| Structure | Training complexity | Inference per-step | KV/state growth | Representative impl |
|:---|:---|:---|:---|:---|
| **Softmax Attention** | $O(N^2 d)$ | $O(N d)$ (KV cache grows linearly) | KV cache grows linearly with the sequence | GPT, Llama |
| **Linear Attention (vectorized)** | $O(N d^2)$ | $O(d^2)$ (recurrent form) | Fixed-size state | Katharopoulos 2020 |
| **SSM / Mamba-1** | $O(N d^2)$ (parallel scan) | $O(d^2)$ (recurrent form) | Fixed-size state | Mamba, S4 |
| **Mamba-2 (SSD)** | $O(N d^2)$ (matrix form, parallel) | $O(d^2)$ (recurrent form) | Fixed-size state | Mamba-2 |
| **Lightning Attention** | $O(N d^2)$ (chunk-wise) | $O(d^2)$ (recurrent decode) | Fixed-size state | MiniMax-01 |
| **Hybrid** | Between $O(N d^2)$ and $O(N^2 d)$ | Between the two | Mixed | Jamba, Zamba |

Three key observations:

- **Training**: except for softmax attention, all structures have complexity linear in N. But actual speed also depends on GPU utilization — Lightning Attention's chunk-wise design exists to keep the GPU busy.
- **Inference**: softmax attention's KV cache grows linearly with the sequence (memory explosion at long context); the others all use a fixed-size state — the fundamental advantage of SSMs / Linear Attention for long-context inference.
- **Hybrid is an engineering trade-off**: not purely optimal in complexity, but a balance between retrieval precision and long-context efficiency. In production, "absolute optimum" almost never exists.

That wraps up the first route around $O(N^2)$. The next route, Sparse Attention, does not touch softmax — it attacks from another angle: only compute the important query-key pairs.


How should they be combined? Use softmax layers where exact retrieval is essential, and linear or SSM layers where very long context must fit a budget. Production models are commonly hybrid: a few softmax layers preserve retrieval quality while many linear layers extend context. NSA and DSA represent another option in which the model learns sparsity rather than following fixed rules.


Use this checklist to connect the evolution:

- **Original bill:** $N \times N$ token pairs; moving from 1K to 1M context multiplies the matrix one million times.
- **Sparse Attention (2020):** compute fewer pairs. Fixed windows are efficient but human-designed; NSA/DSA learn the sparse pattern.
- **Linear Attention (2020):** reorder multiplication so the intermediate is $d \times d$ rather than $N \times N$; exact retrieval weakens without sharp softmax.
- **SSM / Mamba (2021–2023):** modern recurrent models with constant state and parallel training; selectivity adds content-dependent memory.
- **Convergence (since 2024):** Mamba-2 connects SSMs and Linear Attention; hybrid models combine their strengths with softmax.
- **Choice:** softmax for exact retrieval, linear/SSM for long-context efficiency, and usually a hybrid in production.


The three exercises correspond to the three main mechanisms.

> You may ask AI for hints or help breaking down steps, but avoid asking it to complete the exercises.


### Exercise 1: Linear Attention Complexity

Hint: calculate FLOPs for $\phi(K)^T V$ (order $Nd^2$), then for $Q(\phi(K)^T V)$, and compare with the $N^2d$ cost of $QK^T$.


In [ ]:
# Exercise 1: calculate complexity

N = 8192
d = 64

# TODO: fill in the calculations
softmax_flops = 2 * N * N * d
linear_flops = 2 * N * d * d
ratio = softmax_flops / linear_flops

assert softmax_flops is not None, "Calculate softmax_flops first"
assert linear_flops is not None, "Calculate linear_flops first"
assert ratio is not None, "Calculate ratio first"

expected_softmax = 2 * N * N * d
expected_linear = 2 * N * d * d
expected_ratio = expected_softmax / expected_linear

assert softmax_flops == expected_softmax
assert linear_flops == expected_linear
assert abs(ratio - expected_ratio) < 0.001

print("Exercise 1 passed")
print(f"   Softmax attention FLOPs: {softmax_flops:,} ({softmax_flops:.2e})")
print(f"   Linear attention FLOPs:  {linear_flops:,} ({linear_flops:.2e})")
print(f"   Ratio: {ratio:.0f}x, so linear attention uses {ratio:.0f}x less computation")
print("   Key observation: when N is much larger than d, linear attention saves substantial computation")


In [ ]:
# Exercise 1: implement the vectorized form of linear attention

import torch
import torch.nn.functional as F

def linear_attention_vectorized(Q, K, V):
    """Vectorized linear attention with O(Nd^2) complexity.

    Use associativity: first compute phi(K)^T @ V, a d-by-d matrix,
    then multiply it by phi(Q). Here phi(x) = elu(x) + 1.
    """
    # TODO: complete the following lines
    phi_Q = F.elu(Q) + 1
    phi_K = F.elu(K) + 1
    KV = phi_K.T @ V
    out = phi_Q @ KV
    # Normalize each row by its total score
    normalizer = (phi_Q @ phi_K.sum(dim=0, keepdim=True).T).clamp(min=1e-6)
    return out / normalizer

# Verify the implementation
torch.manual_seed(42)
N, d = 16, 8
Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

out = linear_attention_vectorized(Q, K, V)

# Compare with a direct O(N^2 d) implementation
def linear_attention_naive(Q, K, V):
    phi_Q = F.elu(Q) + 1
    phi_K = F.elu(K) + 1
    scores = phi_Q @ phi_K.T
    return (scores @ V) / scores.sum(dim=-1, keepdim=True)

out_ref = linear_attention_naive(Q, K, V)
assert out.shape == (N, d), f"Expected output shape ({N}, {d}), got {out.shape}"
assert torch.allclose(out, out_ref, atol=1e-5), "The result differs from the direct implementation"

print("Exercise 1 passed")
print(f"   Output shape: {out.shape}")
print(f"   Matches the direct implementation: {torch.allclose(out, out_ref, atol=1e-5)}")


### Exercise 2: Sliding-Window Mask

Hint: begin with a causal mask, then set positions whose column distance exceeds `window_size` to $-\infty$.


In [ ]:
# Exercise 2: implement a sliding-window mask

import torch

def make_sliding_window_mask(N, W):
    """Construct a causal sliding-window mask.

    Return an [N, N] tensor where 0 means visible and -inf means blocked.
    """
    # 1. Causal mask: fill the upper triangle above the diagonal with -inf
    rows = torch.arange(N).unsqueeze(1)
    cols = torch.arange(N).unsqueeze(0)
    mask = torch.zeros(N, N)
    mask = mask.masked_fill(cols > rows, float('-inf'))
    # 2. Sliding window: block positions whose distance exceeds W - 1
    mask = mask.masked_fill((rows - cols) > (W - 1), float('-inf'))
    return mask


# Verify the implementation
mask = make_sliding_window_mask(N=8, W=4)

# Check the basic properties
assert mask.shape == (8, 8), f"Expected shape (8, 8), got {mask.shape}"
# A token should see itself
assert mask[0, 0] == 0, "(0, 0) should be visible"
# Distance 3 is inside a four-token window, whose maximum distance is W - 1
assert mask[3, 0] == 0, "(3, 0) is at distance 3 and should be visible for W=4"
# Distance 4 is outside the window
assert mask[4, 0] == float('-inf'), "(4, 0) is outside a window of W=4 and should be blocked"
# Future positions must be blocked by the causal mask
assert mask[0, 1] == float('-inf'), "(0, 1) is in the future and should be blocked"

# Visualize the mask
import matplotlib.pyplot as plt
plt.figure(figsize=(5, 5))
plt.imshow(mask, cmap='RdYlGn', vmin=-10, vmax=0)
plt.colorbar(label='mask value')
plt.xlabel('Key position')
plt.ylabel('Query position')
plt.title(f'Sliding window mask (N=8, W=4)\ngreen=visible, red=blocked')
plt.tight_layout()
plt.show()

print("Exercise 2 passed")
print("   The sliding-window mask is correct")


### Exercise 3: Layer Types in a Hybrid Architecture

Hint: Linear Attention loses softmax sharpness, while exact retrieval depends on that sharpness. What purpose does the single softmax layer serve?


In [ ]:
# Exercise 3: layer-type distribution in a hybrid architecture

num_layers = 32
period = 4  # The fourth layer in every group of four uses softmax attention

# TODO: complete the calculations
layer_types = ['softmax' if (i + 1) % period == 0 else 'linear' for i in range(num_layers)]
n_softmax = sum(1 for t in layer_types if t == 'softmax')
softmax_ratio = n_softmax / num_layers

assert layer_types is not None
assert n_softmax is not None
assert softmax_ratio is not None

expected_types = ['softmax' if (i+1) % period == 0 else 'linear' for i in range(num_layers)]
expected_n = sum(1 for t in expected_types if t == 'softmax')
expected_ratio = expected_n / num_layers

assert layer_types == expected_types
assert n_softmax == expected_n
assert abs(softmax_ratio - expected_ratio) < 0.001

print("Exercise 3 passed")
print(f"   Total layers: {num_layers}")
print(f"   Softmax layers: {n_softmax} ({softmax_ratio*100:.0f}%)")
print(f"   Linear layers: {num_layers - n_softmax} ({(1-softmax_ratio)*100:.0f}%)")
print("   Key observation: even 25% softmax layers are usually enough to preserve retrieval ability")
